In [1]:
from leakly import SimulationConfig, simulate_dataset

simulated = simulate_dataset(
    SimulationConfig(
        n_samples=200,
        n_features=100,
        n_covariates=3,
        effect_fraction=0.1,
        effect_size=0.5,
        class_balance=0.5,
        random_state=42,
    )
)

In [2]:
from leakly import FeatureSelectionConfig, feature_selection

selected_features, selected_indices = feature_selection(
    simulated.X, 
    simulated.y, 
    simulated.covariates, 
    feature_names=simulated.feature_names,
    config=FeatureSelectionConfig(
        method="LinearRegressionDAA",
        alpha=0.05,
        correction_method="fdr_bh",
        minimum_effect_size=None,
        top_ranks=None,
    ),
)
print("Selected features:", selected_features)

LinearRegressionDAA:   0%|          | 0/100 [00:00<?, ?feature/s]

Selected features: ['feature_10', 'feature_8', 'feature_6', 'feature_4', 'feature_7', 'feature_2']


In [3]:
X, y, covariates = simulated.X, simulated.y, simulated.covariates
X = X[:, selected_indices]
print("Shape of selected feature matrix:", X.shape)

# randomly shuffle (X, y, covariates) for splitting train/test sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test, covariates_train, covariates_test = train_test_split(
    X, y, covariates, test_size=0.2, random_state=42
)

from leakly import MLConfig, ml_model
test_auc = ml_model(
    X_train, 
    y_train, 
    X_test,
    y_test,
    config=MLConfig(
        model="random_forest",
        problem_type="binary_classification",
        metric="auc",
        random_state=42,
        model_params={"n_estimators": 100, "max_depth": 5},
    ),
)

print("Test AUC:", test_auc)


Shape of selected feature matrix: (200, 6)
Test AUC: 0.8674999999999999
